# LSTM Model — Bihar Flood Prediction
LSTM captures temporal dependencies. We reshape features as timesteps so LSTM can learn sequential patterns across the feature space (especially useful with year/month/day features).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

## 1. Load & Preprocess Data

In [ ]:
df = pd.read_csv('bihar_flood_data.csv')

df['Date'] = pd.to_datetime(df['Date'])
df['year']       = df['Date'].dt.year
df['month']      = df['Date'].dt.month
df['day']        = df['Date'].dt.day
df['is_monsoon'] = df['month'].apply(lambda x: 1 if x in [6,7,8,9] else 0)
df.drop(columns=['Date'], inplace=True)
df = pd.get_dummies(df, columns=['District'], drop_first=True)

X = df.drop(columns=['Flood']).values
y = df['Flood'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# LSTM input: (samples, timesteps, features)
# Each feature is treated as one timestep with 1 feature value
X_train_lstm = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_lstm  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print('LSTM Input shape — Train:', X_train_lstm.shape, '| Test:', X_test_lstm.shape)

## 2. Build Bidirectional LSTM Model

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))

timesteps = X_train_lstm.shape[1]

model = Sequential([
    # Bidirectional LSTM reads sequence forward & backward
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(timesteps, 1)),
    BatchNormalization(),
    Dropout(0.3),

    Bidirectional(LSTM(64, return_sequences=True)),
    BatchNormalization(),
    Dropout(0.3),

    Bidirectional(LSTM(32, return_sequences=False)),
    BatchNormalization(),
    Dropout(0.2),

    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 3. Train Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train_lstm, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss Curve — BiLSTM')
axes[0].legend()
axes[1].plot(history.history['accuracy'], label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Accuracy Curve — BiLSTM')
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Evaluate Model

In [ ]:
y_pred_prob = model.predict(X_test_lstm).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)

print(f'Accuracy : {acc:.4f}')
print(f'F1 Score : {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Flood','Flood']))

import json
with open('dl_lstm_results.json', 'w') as fp:
    json.dump({'model': 'BiLSTM', 'accuracy': round(acc,4), 'f1': round(f1,4)}, fp)
print('Results saved to dl_lstm_results.json')

## 6. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No Flood','Flood'],
                                         cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix — BiLSTM')
RocCurveDisplay.from_predictions(y_test, y_pred_prob, ax=axes[1])
axes[1].set_title('ROC Curve — BiLSTM')
plt.tight_layout()
plt.show()